# Proyecto Waze
## Inspección y preparación de datos de abandono
**Hernán Licastro | Curso 1 | Proyecto educativo**

Datos sintéticos y escenario ficticio. No representan usuarios reales de Waze.

El equipo requiere inspeccionar y organizar los datos antes del EDA. Se revisan la estructura, la calidad y las diferencias descriptivas entre usuarios.

## Objetivos
Importar e inspeccionar los datos, comparar grupos, construir indicadores y comunicar hallazgos mediante PACE. No se entrena un modelo ni se realizan pruebas de hipótesis.

## Tipos de datos y resumen

## Etapas PACE

Planificar, Analizar, Construir y Ejecutar.

## PACE Planificar

### Tarea 1 Comprender la situación
¿Cómo prepararse para comprender y organizar los datos?


Para comprender y organizar los datos, comenzaré revisando el objetivo del proyecto y el diccionario de variables. Identificaré qué representa cada fila, cómo se define el abandono y qué período cubre cada variable. Luego utilizaré Python y pandas para inspeccionar las primeras filas, verificar los tipos de datos y detectar valores faltantes, duplicados y posibles inconsistencias. Revisaré las estadísticas descriptivas y compararé inicialmente a los usuarios que abandonaron con quienes permanecieron. Documentaré los hallazgos y las limitaciones para preparar el posterior análisis exploratorio.

## PACE Analizar

### Tarea 2a Importación de paquetes

In [1]:
# Import packages for data manipulation

import pandas as pd

import numpy as np

Cargar el CSV original en un DataFrame. Las rutas permiten ejecutar desde la raíz del proyecto o desde notebooks.

In [2]:
from pathlib import Path
paths = [Path("../data/waze_dataset.csv"), Path("data/waze_dataset.csv"), Path("waze_dataset.csv")]
path = next((p for p in paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("No se encontró waze_dataset.csv")
df = pd.read_csv(path)

### Tarea 2b Resumen
Inspeccionar las primeras diez filas y la estructura. Identificar dimensiones, tipos y faltantes.

In [3]:

df.head(10)

,ID,label,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days,device
0,0,retained,283,226,296.748273,2276,208,0,2628.845068,1985.775061,28,19,Android
1,1,retained,133,107,326.896596,1225,19,64,13715.920550,3160.472914,13,11,iPhone
2,2,retained,114,95,135.522926,2651,0,0,3059.148818,1610.735904,14,8,Android
3,3,retained,49,40,67.589221,15,322,7,913.591123,587.196542,7,3,iPhone
4,4,retained,84,68,168.247020,1562,166,5,3950.202008,1219.555924,27,18,Android
5,5,retained,113,103,279.544437,2637,0,0,901.238699,439.101397,15,11,iPhone
6,6,retained,3,2,236.725314,360,185,18,5249.172828,726.577205,28,23,iPhone
7,7,retained,39,35,176.072845,2999,0,0,7892.052468,2466.981741,22,20,iPhone
8,8,retained,57,46,183.532018,424,0,26,2651.709764,1594.342984,25,20,Android
9,9,churned,84,68,244.802115,2997,72,0,6043.460295,2341.838528,7,3,iPhone


In [4]:

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       14999 non-null  int64  
 1   label                    14299 non-null  object 
 2   sessions                 14999 non-null  int64  
 3   drives                   14999 non-null  int64  
 4   total_sessions           14999 non-null  float64
 5   n_days_after_onboarding  14999 non-null  int64  
 6   total_navigations_fav1   14999 non-null  int64  
 7   total_navigations_fav2   14999 non-null  int64  
 8   driven_km_drives         14999 non-null  float64
 9   duration_minutes_drives  14999 non-null  float64
 10  activity_days            14999 non-null  int64  
 11  driving_days             14999 non-null  int64  
 12  device                   14999 non-null  object 
dtypes: float64(3), int64(8), object(2)
memory usage: 1.5+ MB



1)En las primeras 10 filas no se observan valores faltantes. Sin embargo, esa muestra no permite descartar que existan en el resto del conjunto.

2)El conjunto contiene 14.999 filas y 13 columnas. Hay ocho columnas de tipo int64, tres de tipo float64 y dos de tipo object: label y device. Cada fila representa un usuario según la documentación del proyecto.

3)Sí, existen 700 valores faltantes, todos en la columna label, que representan aproximadamente el 4,67 % de las filas. Las otras 12 columnas están completas. Una etiqueta faltante significa que desconocemos la clasificación del usuario; no debemos interpretarla como abandono ni como permanencia.

### Tarea 2c Valores faltantes y estadísticas
Comparar las 700 filas sin etiqueta con las filas completas. ¿Se observan diferencias?

In [5]:
# Isolate rows with null values

df_null = df[df["label"].isna()]

# Display summary stats of rows with null values

df_null.describe()

,ID,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days
count,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000,700.000000
mean,7405.584286,80.837143,67.798571,198.483348,1709.295714,118.717143,30.371429,3935.967029,1795.123358,15.382857,12.125714
std,4306.900234,79.987440,65.271926,140.561715,1005.306562,156.308140,46.306984,2443.107121,1419.242246,8.772714,7.626373
min,77.000000,0.000000,0.000000,5.582648,16.000000,0.000000,0.000000,290.119811,66.588493,0.000000,0.000000
25%,3744.500000,23.000000,20.000000,94.056340,869.000000,4.000000,0.000000,2119.344818,779.009271,8.000000,6.000000
50%,7443.000000,56.000000,47.500000,177.255925,1650.500000,62.500000,10.000000,3421.156721,1414.966279,15.000000,12.000000
75%,11007.000000,112.250000,94.000000,266.058022,2508.750000,169.250000,43.000000,5166.097373,2443.955404,23.000000,18.000000
max,14993.000000,556.000000,445.000000,1076.879741,3498.000000,1096.000000,352.000000,15135.391280,9746.253023,31.000000,30.000000


In [6]:
# Isolate rows without null values

df_not_null = df[df["label"].notna()]

# Display summary stats of rows without null values

df_not_null.describe()

,ID,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days
count,14299.000000,14299.000000,14299.000000,14299.000000,14299.000000,14299.000000,14299.000000,14299.000000,14299.000000,14299.000000,14299.000000
mean,7503.573117,80.623820,67.255822,189.547409,1751.822505,121.747395,29.638296,4044.401535,1864.199794,15.544653,12.182530
std,4331.207621,80.736502,65.947295,136.189764,1008.663834,147.713428,45.350890,2504.977970,1448.005047,9.016088,7.833835
min,0.000000,0.000000,0.000000,0.220211,4.000000,0.000000,0.000000,60.441250,18.282082,0.000000,0.000000
25%,3749.500000,23.000000,20.000000,90.457733,878.500000,10.000000,0.000000,2217.319909,840.181344,8.000000,5.000000
50%,7504.000000,56.000000,48.000000,158.718571,1749.000000,71.000000,9.000000,3496.545617,1479.394387,16.000000,12.000000
75%,11257.500000,111.000000,93.000000,253.540450,2627.500000,178.000000,43.000000,5299.972162,2466.928876,23.000000,19.000000
max,14998.000000,743.000000,596.000000,1216.154633,3500.000000,1236.000000,415.000000,21183.401890,15851.727160,31.000000,30.000000



Las estadísticas descriptivas de los usuarios con y sin etiqueta son similares en términos generales. Por ejemplo, ambos grupos tienen una mediana de 56 sesiones mensuales y 12 días de conducción. La mediana de viajes es de 47,5 para los usuarios sin etiqueta y de 48 para quienes tienen etiqueta.

Se observan algunas diferencias: el grupo sin etiqueta presenta una mayor mediana de sesiones totales estimadas y una menor mediana de antigüedad. Sin embargo, esta comparación descriptiva no muestra un patrón claramente diferenciado ni permite concluir que la ausencia de etiquetas sea completamente aleatoria. Es necesario continuar investigando los valores faltantes antes de decidir su tratamiento.

### Tarea 2d Dispositivos
¿Cuántos usuarios de cada dispositivo tienen etiqueta faltante?

In [7]:
# Get count of null values by device

df_null["device"].value_counts()

device
iPhone     447
Android    253
Name: count, dtype: int64


De los 700 registros con valores faltantes en label, 447 corresponden a usuarios de iPhone y 253 a usuarios de Android.

Calcular porcentajes de dispositivos entre las filas sin etiqueta.

In [8]:
# Calculate % of iPhone nulls and Android nulls

# Porcentaje de cada dispositivo entre los usuarios sin etiqueta
df_null["device"].value_counts(normalize=True) * 100

device
iPhone     63.857143
Android    36.142857
Name: proportion, dtype: float64

Comparar con la distribución del conjunto completo.

In [9]:
# Calculate % of iPhone users and Android users in full dataset

# Porcentaje de cada dispositivo en el conjunto completo
df["device"].value_counts(normalize=True) * 100

device
iPhone     64.484299
Android    35.515701
Name: proportion, dtype: float64

La composición por dispositivo es similar: iPhone 63,86 % entre registros sin etiqueta frente a 64,48 % del total; Android 36,14 % frente a 35,52 %. Esto no demuestra que los valores falten completamente al azar.

### Distribución de abandono y permanencia
Calcular cantidades y porcentajes entre las etiquetas conocidas.

In [10]:
# Calculate counts of churned vs. retained

# Contar usuarios con etiqueta: permanencia y abandono
counts = df["label"].value_counts()

# Calcular porcentajes entre los usuarios con etiqueta
percentages = df["label"].value_counts(normalize=True) * 100

# Mostrar ambos resultados en una tabla
pd.DataFrame({
    "cantidad": counts,
    "porcentaje": percentages
})

,cantidad,porcentaje
label,,
retained,11763,82.264494
churned,2536,17.735506


El 82,26 % de las etiquetas conocidas corresponde a permanencia y el 17,74 % a abandono. Comparar medianas de las variables numéricas. ID es un identificador y su mediana no describe comportamiento.

In [11]:
# Calculate median values of all columns for churned and retained users

# Comparar las medianas de las variables numéricas según la etiqueta
df.groupby("label").median(numeric_only=True)

,ID,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days
label,,,,,,,,,,,
churned,7477.5,59.0,50.0,164.339042,1321.0,84.5,11.0,3652.655666,1607.183785,8.0,6.0
retained,7509.0,56.0,47.0,157.586756,1843.0,68.0,9.0,3464.684614,1458.046141,17.0,14.0


Quienes abandonaron tienen medianas de 50 viajes y 8 días activos; quienes permanecieron, 47 viajes y 17 días. Las medianas de distancia mensual son 3.652,66 y 3.464,68 km. Estas asociaciones no prueban causalidad.

### Kilómetros por viaje
Calcular el cociente de cada usuario y luego su mediana por grupo. Reemplazar denominadores cero por NaN.

In [12]:
# Add a column to df called `km_per_drive`
# Crear una columna con los kilómetros por viaje
# Reemplazar los denominadores iguales a cero por NaN para evitar dividir por cero
df["km_per_drive"] = (
    df["driven_km_drives"] / df["drives"].replace(0, np.nan)
)



# Group by `label`, calculate the median, and isolate for km per drive

# Calcular la mediana de kilómetros por viaje para cada grupo
df.groupby("label")["km_per_drive"].median()

label
churned     73.491807
retained    74.051037
Name: km_per_drive, dtype: float64

Las medianas de km por viaje son similares: 73,49 para abandono y 74,05 para permanencia. Calcular ahora los kilómetros por día de conducción.

In [13]:
# Add a column to df called `km_per_driving_day`

# Calcular kilómetros por día de conducción para cada usuario
df["km_per_driving_day"] = (
    df["driven_km_drives"] / df["driving_days"].replace(0, np.nan)
)

# Group by `label`, calculate the median, and isolate for km per driving day

# Obtener la mediana para cada grupo
df.groupby("label")["km_per_driving_day"].median()

label
churned     523.086749
retained    272.628549
Name: km_per_driving_day, dtype: float64

Calcular viajes por día de conducción y su mediana por etiqueta.

In [14]:
# Add a column to df called `drives_per_driving_day`

# Calcular viajes por día de conducción para cada usuario
df["drives_per_driving_day"] = (
    df["drives"] / df["driving_days"].replace(0, np.nan)
)

# Group by `label`, calculate the median, and isolate for drives per driving day

# Obtener la mediana para cada grupo
df.groupby("label")["drives_per_driving_day"].median()

label
churned     7.454545
retained    3.750000
Name: drives_per_driving_day, dtype: float64

Las medianas válidas de km por día son 523,09 frente a 272,63; las de viajes por día, 7,45 frente a 3,75. Se excluyen denominadores cero, por lo que el resultado difiere de los 698 km del texto orientativo original. Los datos no permiten inferir la ocupación de los usuarios ni las causas del abandono.

### Comparación por dispositivo
Contar dispositivos dentro de cada etiqueta.

In [15]:
# For each label, calculate the number of Android users and iPhone users

# Contar usuarios por etiqueta y dispositivo
df.groupby("label")["device"].value_counts()

label     device 
churned   iPhone     1645
          Android     891
retained  iPhone     7580
          Android    4183
Name: count, dtype: int64

Calcular porcentajes de dispositivos dentro de cada etiqueta.

In [16]:
# For each label, calculate the percentage of Android users and iPhone users

# Calcular la distribución porcentual de dispositivos por etiqueta
df.groupby("label")["device"].value_counts(normalize=True) * 100

label     device 
churned   iPhone     64.865931
          Android    35.134069
retained  iPhone     64.439344
          Android    35.560656
Name: proportion, dtype: float64

La composición por dispositivo es similar. Para calcular tasas de abandono dentro de cada dispositivo, normalizar la tabla cruzada por filas.

In [17]:
# Calcular porcentajes de abandono y permanencia dentro de cada dispositivo
pd.crosstab(
    df["device"],
    df["label"],
    normalize="index"
) * 100

label,churned,retained
device,,
Android,17.560110,82.439890
iPhone,17.831978,82.168022


## PACE Construir
No aplica a esta entrega. No se entrena un modelo.

## PACE Ejecutar

### Tarea 3 Conclusiones
Responder las seis preguntas sobre faltantes, mediana, dudas pendientes, dispositivos, características de los grupos y tasas de abandono.

1)¿Había valores faltantes? ¿Cuántos y en qué variables? ¿Se observó algún patrón?

El conjunto contiene 700 valores faltantes, todos en label, equivalentes al 4,67 % de los 14.999 registros. Las demás columnas están completas. Los registros con y sin etiqueta presentan estadísticas descriptivas generalmente similares. Además, la distribución de dispositivos entre los registros sin etiqueta es cercana a la del conjunto completo. No se identificó un patrón evidente en estas comparaciones, aunque esto no demuestra que los valores falten de forma completamente aleatoria.

2)¿Qué ventaja tiene utilizar la mediana en lugar de la media?

La mediana es menos sensible a los valores extremos y permite describir mejor el centro de una distribución asimétrica. Esto resulta útil para variables como kilómetros recorridos y duración de conducción, donde algunos usuarios presentan valores muy elevados que pueden aumentar considerablemente la media.

3)¿Qué nuevas preguntas surgieron para investigar o consultar al equipo de Waze?

Sería necesario aclarar cómo se define y registra el abandono, por qué faltan 700 etiquetas y qué período exacto cubren las variables. También convendría investigar los registros con cero días de conducción pero kilómetros o viajes positivos, y validar los valores elevados de distancia y duración. Finalmente, sería útil explorar por qué quienes abandonaron registran menos días de actividad y menor antigüedad, y verificar que las variables utilizadas en un futuro modelo estén disponibles antes del período que se desea predecir.

4)¿Qué porcentaje de los usuarios utiliza Android y qué porcentaje utiliza iPhone?

Sobre el conjunto completo de 14.999 usuarios, el 35,52 % utiliza Android y el 64,48 % utiliza iPhone.

5)¿Qué características distinguen a quienes abandonaron de quienes permanecieron?

Entre los 14.299 usuarios con etiqueta conocida, el 17,74 % abandonó y el 82,26 % permaneció. Quienes abandonaron presentan menos días de actividad mensual —mediana de 8 frente a 17— y menos días de conducción —6 frente a 14—. También tienen menor antigüedad: 1.321 frente a 1.843 días.

Sin embargo, registran una cantidad ligeramente mayor de viajes mensuales y kilómetros recorridos. Al excluir del cálculo los denominadores iguales a cero, la mediana de kilómetros por día de conducción es de 523,09 para quienes abandonaron y 272,63 para quienes permanecieron; la mediana de viajes por día de conducción es de 7,45 y 3,75, respectivamente. Los kilómetros por viaje son similares: 73,49 y 74,05.

Estos resultados sugieren un uso más concentrado en menos días entre quienes abandonaron. Son asociaciones descriptivas y no permiten establecer las causas del abandono ni identificar la ocupación de los usuarios.

6)¿Existe una diferencia apreciable en la tasa de abandono entre usuarios de iPhone y Android?

Entre los usuarios con etiqueta conocida, la tasa de abandono es del 17,83 % para iPhone y del 17,56 % para Android. La diferencia de aproximadamente 0,27 puntos porcentuales es pequeña en términos descriptivos. Este análisis no muestra una diferencia marcada por dispositivo, aunque no se realizó una prueba estadística para evaluar su significación.

## Cierre
La inspección prepara el EDA. Acordar el tratamiento de faltantes y las ventanas temporales antes del modelado.

## Calidad y cobertura de los cocientes
Se conservan todas las filas originales y no se imputan etiquetas.

In [18]:
raw = df.iloc[:, :13]
pd.Series({
    "rows": len(raw), "columns": raw.shape[1],
    "duplicate_rows": int(raw.duplicated().sum()),
    "duplicate_ids": int(raw.ID.duplicated().sum()),
    "missing_labels": int(raw.label.isna().sum()),
    "zero_drives_positive_km": int(((raw.drives == 0) & (raw.driven_km_drives > 0)).sum()),
    "zero_driving_days_positive_km": int(((raw.driving_days == 0) & (raw.driven_km_drives > 0)).sum())
})

rows                             14999
columns                             13
duplicate_rows                       0
duplicate_ids                        0
missing_labels                     700
zero_drives_positive_km            106
zero_driving_days_positive_km     1024
dtype: int64

In [19]:
df["km_per_drive"] = df["driven_km_drives"] / df["drives"].replace(0, np.nan)
df["km_per_driving_day"] = df["driven_km_drives"] / df["driving_days"].replace(0, np.nan)
df["drives_per_driving_day"] = df["drives"] / df["driving_days"].replace(0, np.nan)
df.groupby("label")[["km_per_drive", "km_per_driving_day", "drives_per_driving_day"]].agg(["count", "median"])

km_per_drive             ... drives_per_driving_day          
                count     median  ...                  count    median
label                             ...                                 
churned          2522  73.491807  ...                   2143  7.454545
retained        11675  74.051037  ...                  11173  3.750000

[2 rows x 6 columns]

Hay 106 registros con cero viajes y km positivos, y 1.024 con cero días de conducción y km positivos. Para los cocientes por día quedan 2.143 de 2.536 usuarios con abandono y 11.173 de 11.763 con permanencia. Se excluyen del cociente 393 (15,50 %) y 590 (5,02 %), respectivamente. La cobertura desigual limita la comparación.

Las celdas se ejecutaron secuencialmente en un proceso Python nuevo. No se validó mediante un kernel Jupyter en este entorno.